<a href="https://colab.research.google.com/github/BrunoStrufaldi/Automa-o-GoogleColab/blob/main/WebScraping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Remove instalações quebradas ou antigas
!apt-get remove -y --purge chromium-browser chromium-chromedriver
!rm -rf google-chrome-stable_current_amd64.deb

# 2. Instala dependências do sistema e o Selenium
!apt-get update
!apt-get install -y wget
!pip install selenium webdriver-manager

# 3. Baixa e instala o Google Chrome Oficial (.deb)
!wget https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt install -y ./google-chrome-stable_current_amd64.deb

# 4. Configuração do Driver (Selenium Manager fará o trabalho pesado)
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import time
import pandas as pd

# Configurações para rodar no Colab sem tela
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-gpu')
options.add_argument('--window-size=1920,1080')

# Inicializa o driver (O Selenium 4 baixa o driver compatível automaticamente agora)
try:
    print("Iniciando o Google Chrome...")
    driver = webdriver.Chrome(options=options)
    print("Sucesso! Driver iniciado.")
except Exception as e:
    print(f"Erro fatal: {e}")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
Package 'chromium-browser' is not installed, so not removed
Package 'chromium-chromedriver' is not installed, so not removed
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64

In [3]:
# Verifica se o driver existe
if 'driver' not in locals():
    print("ERRO: O driver não foi iniciado. Rode a Célula 1 novamente.")
else:
    url = "https://www.distribuidorawi.com.br/acessorios-para-fogoes/bacias-p-fogoes/"
    driver.get(url)
    print(f"Acessando: {url}")
    time.sleep(3) # Espera inicial generosa

    # --- 1. Lidar com o Banner de Cookies ---
    try:
        cookie_btn = driver.find_element(By.XPATH, "//button[contains(text(), 'Entendi')] | //a[contains(text(), 'Entendi')]")
        driver.execute_script("arguments[0].click();", cookie_btn)
        print("Banner de cookies fechado.")
        time.sleep(1)
    except:
        pass # Segue o jogo se não achar

    # --- 2. Loop Baseado em Quantidade de Produtos ---
    print("Iniciando varredura agressiva...")

    last_count = 0
    strikes = 0 # Contador de tentativas falhas
    max_strikes = 3 # Se falhar 3 vezes seguidas em achar produtos novos, encerra

    while strikes < max_strikes:
        # 1. Rola até o fim da página
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(2)

        # 2. Tenta encontrar e clicar no botão "Mostrar mais" (Visível ou não)
        try:
            # Busca genérica pelo botão de carga
            load_btns = driver.find_elements(By.XPATH, "//*[contains(text(), 'Mostrar mais produtos')] | //a[contains(@class, 'js-load-more')]")

            clicked = False
            for btn in load_btns:
                try:
                    # Tenta clicar via JavaScript em TODOS os botões candidatos encontrados
                    driver.execute_script("arguments[0].click();", btn)
                    clicked = True
                except:
                    pass

            if clicked:
                print(" (clique enviado)", end="")
                time.sleep(3) # Tempo para o site responder
            else:
                # Se não achou botão, apenas espera (pode ser carregamento automático por scroll)
                time.sleep(2)

        except Exception as e:
            pass

        # 3. Conta quantos produtos temos agora na tela
        soup = BeautifulSoup(driver.page_source, 'html.parser')
        # Tenta os dois tipos de seletores comuns
        current_products = soup.find_all('div', class_='item-product')
        if not current_products:
            current_products = soup.find_all('div', {'data-store': 'product-item-info'})

        current_count = len(current_products)
        print(f"\nProdutos encontrados agora: {current_count}")

        # 4. Verifica se aumentou
        if current_count > last_count:
            print(f"-> Sucesso! Novos produtos carregados (+{current_count - last_count}). Zerando contagem de erros.")
            last_count = current_count
            strikes = 0 # Reseta os erros, pois funcionou
        else:
            strikes += 1
            print(f"-> Nenhum produto novo ({strikes}/{max_strikes} tentativas de encerrar).")
            # Tenta um scroll levemente para cima para destravar gatilhos de scroll
            driver.execute_script("window.scrollBy(0, -200);")
            time.sleep(1)

    print("\nVarredura encerrada.")

    # --- 3. Extração Final ---
    data = []
    # Re-parseia o HTML final
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    products = soup.find_all('div', class_='item-product')
    if not products:
        products = soup.find_all('div', {'data-store': 'product-item-info'})

    for product in products:
        try:
            name_tag = product.find('div', class_='item-name') or product.find('h3')
            name = name_tag.get_text(strip=True) if name_tag else "Sem Nome"

            price_tag = product.find('span', class_='item-price') or product.find('span', class_='price')
            price = price_tag.get_text(strip=True) if price_tag else "Esgotado/Sem Preço"

            link_tag = product.find('a', href=True)
            link = link_tag['href'] if link_tag else "#"

            data.append({'Produto': name, 'Preço': price, 'Link': link})
        except:
            continue

    # --- 4. Resultado ---
    df = pd.DataFrame(data)
    # Remove duplicatas exatas
    df = df.drop_duplicates(subset=['Produto'])

    print(f"Total Final de Produtos Únicos: {len(df)}")
    display(df)

    # --- 5. Salvar e Baixar a Planilha ---
from google.colab import files

# Define o nome do arquivo
nome_arquivo = 'lista_produtos.xlsx'

# Salva o DataFrame como um arquivo Excel (sem a coluna de índice numérico)
df.to_excel(nome_arquivo, index=False)

# Aciona o download do navegador
files.download(nome_arquivo)
print(f"Download de '{nome_arquivo}' iniciado!")

Acessando: https://www.distribuidorawi.com.br/acessorios-para-fogoes/bacias-p-fogoes/
Banner de cookies fechado.
Iniciando varredura agressiva...
 (clique enviado)
Produtos encontrados agora: 24
-> Sucesso! Novos produtos carregados (+24). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 36
-> Sucesso! Novos produtos carregados (+12). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 48
-> Sucesso! Novos produtos carregados (+12). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 72
-> Sucesso! Novos produtos carregados (+24). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 84
-> Sucesso! Novos produtos carregados (+12). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 96
-> Sucesso! Novos produtos carregados (+12). Zerando contagem de erros.
 (clique enviado)
Produtos encontrados agora: 108
-> Sucesso! Novos produtos carregados (+12). Zerando contagem de erros.
 (cliq

,Produto,Preço,Link
0,521501 BACIA ATLAS /SABAF TRIPLA CHAMA MINI,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/52...
1,101401 - BACIA ECORING TRIPLA CHAMA BIG SUGGAR...,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/10...
2,116904 - BACIA ATLAS SABAF GRANDE AG0588,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/11...
3,117004 - BACIA ATLAS SABAF GRANDE AG0589,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/11...
4,503701 - BACIA FISCHER/BOSCH SABAF PQ S/SAIA Q...,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/50...
...,...,...,...
143,05490 BACIA MUELLER MINI DUPLA CHAMA LAPIDADA ...,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/05...
144,05790 - BACIA BRASTEMP ECO TRIPLA CHAMA ORIG. ...,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/05...
145,03360 - BACIA BRASTEMP CLEAN TRIPLA CHAMA ORIG...,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/03...
146,03348 - BACIA BRASTEMP CLEAN QUADRICHAMA ORIG....,Esgotado/Sem Preço,https://www.distribuidorawi.com.br/produtos/03...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download de 'lista_produtos_sorvetes.xlsx' iniciado!
